# IRT Results Viewer

Load and inspect the tables saved by `IRT/irt.py` or `IRT/run_irt_modal.py`.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

RESULT_KIND = "solver"  # change to "judging" when viewing judging runs

candidate_dirs = [
    Path("IRT/results_modal") / RESULT_KIND,
    Path("IRT/results") / RESULT_KIND,
    Path("results_modal") / RESULT_KIND,
    Path("results") / RESULT_KIND,
    Path("IRT/results_modal"),
    Path("IRT/results"),
    Path("results_modal"),
    Path("results"),
]
RESULT_DIR = next((path for path in candidate_dirs if (path / "run_config.json").exists()), candidate_dirs[0])
print(f"Using results directory: {RESULT_DIR.resolve()}")

In [ ]:
def load_csv(name: str) -> pd.DataFrame | None:
    path = RESULT_DIR / f"{name}.csv"
    if not path.exists():
        print(f"Missing: {path}")
        return None
    return pd.read_csv(path)


def display_table(title: str, df: pd.DataFrame | None, *, sort_by: str | None = None, ascending: bool = False) -> None:
    print(f"\n{title}")
    if df is None:
        return
    view = df.sort_values(sort_by, ascending=ascending).reset_index(drop=True) if sort_by in df.columns else df
    display(view)


config_path = RESULT_DIR / "run_config.json"
run_config = json.loads(config_path.read_text()) if config_path.exists() else {}

capability_scores = load_csv("capability_scores")
capability_scores_with_uncertainty = load_csv("capability_scores_with_uncertainty")
heldout_eval_raw = load_csv("heldout_eval_raw")
heldout_eval_summary = load_csv("heldout_eval_summary")

## Run Config

In [ ]:
run_config

## Capability Scores

In [ ]:
display_table("Capability Scores", capability_scores)

## Capability Scores With Uncertainty

In [ ]:
display_table("Capability Scores With Uncertainty", capability_scores_with_uncertainty)

## Held-Out Model Selection

In [ ]:
display_table("Held-Out Eval Summary", heldout_eval_summary)

In [ ]:
display_table("Held-Out Eval Raw", heldout_eval_raw)

## Compact Views

In [ ]:
if capability_scores is not None:
    ability_cols = [
        "model",
        "accuracy",
        "n_observed",
        "1pl_regular",
        "1pl_item_marginal_mmle",
        "2pl_regular",
        "2pl_item_marginal_mmle",
        "3pl_regular",
        "3pl_item_marginal_mmle",
    ]
    display(capability_scores[[col for col in ability_cols if col in capability_scores.columns]])

In [ ]:
if heldout_eval_summary is not None:
    metric_cols = [
        "fit",
        "heldout_auc_mean",
        "heldout_auc_sd",
        "heldout_log_likelihood_mean",
        "heldout_brier_mean",
        "heldout_ece_mean",
        "n_test_mean",
        "test_positive_rate_mean",
    ]
    display(heldout_eval_summary[[col for col in metric_cols if col in heldout_eval_summary.columns]])